In [ ]:
%pip install azure-ai-contentunderstanding

# Module 3: Content Understanding & Semantic Chunking

In **Module 2**, we solved the "Invisibility" problem: we found the bounding boxes of tables and figures.

But finding a figure isn't enough. A RAG system needs to **understand** what's inside it.
- **Module 1 (Naive)**: Ignored figures completely.
- **Module 2 (Layout)**: Found the box `[x, y, w, h]`.
- **Module 3 (Content Understanding)**: Describing the figure using **GPT-4.1-mini** via Content Understanding, and using document structure for **Semantic Chunking**.

## Objectives
1. **Multimodal Analysis**: Use Content Understanding (powered by GPT-4.1-mini) to generate text descriptions of figures automatically.
2. **Semantic Chunking**: Group text by *Section Headers* instead of arbitrary character counts.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
import pandas as pd
from IPython.display import Image, display, Markdown
import base64

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.contentunderstanding import ContentUnderstandingClient

# Load environment variables
env = load_env()

# --- 1. Setup Document Intelligence (for Structure) ---
doc_endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
credential = DefaultAzureCredential()
doc_client = DocumentIntelligenceClient(endpoint=doc_endpoint, credential=credential)

# --- 2. Setup Azure AI Content Understanding (The "Magic") ---
# Content Understanding often uses a specific regional endpoint (services.ai.azure.com)
# distinct from the generic Cognitive Services endpoint. We switch to the correct one here.
cu_endpoint = doc_endpoint.replace(".cognitiveservices.azure.com", ".services.ai.azure.com")

# Fallback for unexpected URL changes
if "services.ai.azure.com" not in cu_endpoint:
     # If the replacement didn't happen (e.g. custom domain), we might default to hardcoded or original
     # For this workshop, we assume the standard pattern or rely on manual overwrite if needed.
     pass

print(f"Using Content Understanding Endpoint: {cu_endpoint}")
cu_client = ContentUnderstandingClient(endpoint=cu_endpoint, credential=credential)

# --- 3. Setup OpenAI GPT-4o (Fallback/Comparison) ---
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

aoai_client = AzureOpenAI(
    azure_endpoint=env["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview"
)

deployment_name = env.get("AZURE_OPENAI_DEPLOYMENT_GPT4o", "gpt-4o") 

# --- 4. Define Data Paths ---
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "Basic Electrical Engineering R-20.pdf"

print("Clients Initialized (Doc Intel, Content Understanding, OpenAI).")
print(f"Target PDF: {PDF_PATH}")

## Lab 3.1: Azure AI Content Understanding (Real Implementation)

### 🔑 Why Content Understanding, Not Just Document Intelligence?

Both services analyze documents, but serve **different purposes**:

| Aspect | Document Intelligence | Content Understanding |
|--------|----------------------|----------------------|
| **Core Function** | *"What's on the page?"* | *"What does it mean?"* |
| **Figure Output** | Bounding box `[x,y,w,h]` | Bounding box + **AI description** |
| **Chart Output** | Just an image | **Chart.js code** |
| **Diagram Output** | Just an image | **Mermaid.js syntax** |
| **Audio/Video** | ❌ Not supported | ✅ Transcription + analysis |

**Bottom line**: DI tells you *where* things are. CU tells you *what they mean*.

---

### The `prebuilt-documentSearch` Analyzer

This analyzer is optimized for RAG and automated workflows. Key capabilities:

- **Content Analysis**: Text (printed/handwritten), barcodes (12+ types), math formulas (LaTeX), hyperlinks
- **Figure Analysis**: AI descriptions for images, **Chart.js** for charts, **Mermaid.js** for diagrams
- **Structure Analysis**: Paragraphs with contextual roles, complex tables (merged cells, multi-page), hierarchical sections
- **Output**: GitHub Flavored Markdown optimized for LLM comprehension
- **Formats**: PDF, images, Office docs, HTML, Markdown, XML, JSON, CSV, email (EML/MSG)

---

We use the `prebuilt-documentSearch` analyzer below, which performs:
1. **Layout Analysis** – Document structure extraction
2. **Figure Detection & Cropping** – Internal image extraction
3. **Multimodal Description** – Vision AI generates semantic descriptions
4. **Markdown Generation** – LLM-ready output with preserved structure

In [ ]:
# --- REAL WORLD IMPLEMENTATION: CONTENT UNDERSTANDING ---
# Based on: https://github.com/Azure-Samples/azure-ai-content-understanding-python

import requests
import json

print("⚙️ Configuring Content Understanding Resource Defaults...")

# Reset result to avoid stale data from previous runs
result = None

# FORCE API VERSION (GA)
api_version = "2025-11-01"

# --- AUTH & CONFIG CHECK (Proactive) ---
try:
    print("   🔐 Verifying Authentication & Configuration...")
    token_obj = credential.get_token("https://cognitiveservices.azure.com/.default")
    token = token_obj.token
    print("   ✅ Auth Token acquired.")

    # PROACTIVE CONFIGURATION CHECK
    endpoint_clean = cu_endpoint.rstrip("/")
    config_url = f"{endpoint_clean}/contentunderstanding/defaults?api-version={api_version}"
    
    print(f"   🔎 Checking Service Defaults at: {config_url}")
    defaults_resp = requests.get(config_url, headers={"Authorization": f"Bearer {token}"})
    
    should_patch = False
    if defaults_resp.status_code == 200:
        curr_config = defaults_resp.json()
        print(f"   Current Config: {json.dumps(curr_config, indent=2)}")
        
        # Check if required models are mapped
        # CRITICAL: prebuilt-documentSearch requires GPT-4.1-mini, NOT GPT-4.1!
        # See: https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models
        deps = curr_config.get("modelDeployments", {})
        if not deps.get("gpt-4.1-mini"):
            print("   ⚠️ 'gpt-4.1-mini' deployment is MISSING in defaults.")
            print("      👉 prebuilt-documentSearch REQUIRES gpt-4.1-mini!")
            should_patch = True
    else:
        print(f"   ⚠️ Could not read defaults ({defaults_resp.status_code}). Will try to Patch blind.")
        should_patch = True
    
    if should_patch:
        print("\n   🔧 PATCHING Defaults to ensure model linkage...")
        print("      NOTE: prebuilt-documentSearch requires gpt-4.1-mini, NOT gpt-4.1")
        
        # Get deployment names from environment or use conventions
        # The DEPLOYMENT NAME must match actual deployments in Azure AI Foundry
        gpt41_mini_deployment = env.get("GPT_4_1_MINI_DEPLOYMENT", "gpt-4.1-mini")
        embedding_deployment = env.get("TEXT_EMBEDDING_3_LARGE_DEPLOYMENT", "text-embedding-3-large")
        
        config_payload = {
            "modelDeployments": {
                "gpt-4.1-mini": gpt41_mini_deployment,
                "text-embedding-3-large": embedding_deployment
            }
        }
        
        print(f"      Payload: {json.dumps(config_payload)}")
        
        patch_resp = requests.patch(
            config_url, 
            json=config_payload,
            headers={"Authorization": f"Bearer {token}"}
        )
        if patch_resp.status_code in [200, 202]:
            print("   ✅ Patch Successful.")
            # Re-read config to confirm
            verify_resp = requests.get(config_url, headers={"Authorization": f"Bearer {token}"})
            if verify_resp.status_code == 200:
                print(f"   Updated Config: {json.dumps(verify_resp.json(), indent=2)}")
        else:
            print(f"   ❌ Patch Failed: {patch_resp.text}")
            print("\n   ⚠️ ACTION REQUIRED:")
            print("      You must deploy 'gpt-4.1-mini' model in Azure AI Foundry.")
            print("      See: https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models")

except Exception as auth_err:
    print(f"\n❌ AUTHENTICATION/CONFIG CHECK FAILED: {auth_err}")
    print("   Proceeding with analysis attempt anyway (might fail)...")


# --- ANALYSIS ---
analyzer_id = 'prebuilt-documentSearch'

try:
    print(f"\n🔍 Analyzing {PDF_PATH} with Azure AI Content Understanding ({analyzer_id})...")
    with open(PDF_PATH, "rb") as f:
        file_bytes = f.read()

    print(f"   File size: {len(file_bytes)} bytes")

    # Re-init client with explicit API version
    print(f"   Initializing ContentUnderstandingClient with api_version='{api_version}'...")
    cu_client_vnext = ContentUnderstandingClient(
        endpoint=cu_endpoint,
        credential=credential,
        api_version=api_version
    )

    # Try analysis using SDK...
    if hasattr(cu_client_vnext, "begin_analyze_binary"):
        response = cu_client_vnext.begin_analyze_binary(analyzer_id=analyzer_id, binary_input=file_bytes, content_type="application/pdf")
    else:
        response = cu_client_vnext.begin_analyze(analyzer_id=analyzer_id, body=file_bytes, content_type="application/pdf")
    
    print("   Analysis started... waiting for result...")
    result = response.result()

except Exception as e:
    print(f"❌ Analysis Failed: {e}")
    result = None

# --- VALIDATE & FALLBACK ---
has_contents = False

if result:
    # Check for contents in the result
    # The official samples access: result.get("result", {}).get("contents", [])
    # But the SDK returns an object, so we need to handle both cases
    if hasattr(result, "as_dict"):
        res_dict = result.as_dict()
        # Check both "contents" at top level and nested under "result"
        contents_list = res_dict.get("contents") or res_dict.get("result", {}).get("contents", [])
        has_contents = bool(contents_list)
    elif hasattr(result, "contents"):
        has_contents = bool(result.contents)

if result is not None and not has_contents:
    print("\n🧐 TROUBLESHOOTING: 'prebuilt-documentSearch' returned empty contents.")
    print("   This usually means:")
    print("   1. 'gpt-4.1-mini' model is NOT deployed in your Azure AI Foundry project")
    print("   2. The model deployment name doesn't match what's in the defaults config")
    print("   👉 See: https://github.com/Azure-Samples/azure-ai-content-understanding-python#step-2-deploy-required-models")
    
    # Dump full result to file for debugging
    debug_file = "debug_cu_result.json"
    dump_data = result.as_dict() if hasattr(result, "as_dict") else str(result)
    with open(debug_file, "w") as f:
        json.dump(dump_data, f, indent=2, default=str)
    print(f"   👉 Full raw result dumped to: {debug_file}")

# FALLBACK to prebuilt-layout
if result is None or (not has_contents):
    print("\n   🔄 AUTOMATIC FALLBACK to 'prebuilt-layout' (Standard Layout Analysis)...")
    analyzer_id = 'prebuilt-layout'
    try:
        if hasattr(cu_client_vnext, "begin_analyze_binary"):
            response = cu_client_vnext.begin_analyze_binary(analyzer_id=analyzer_id, binary_input=file_bytes, content_type="application/pdf")
        else:
            response = cu_client_vnext.begin_analyze(analyzer_id=analyzer_id, body=file_bytes, content_type="application/pdf")
        result = response.result()
        print("   ✅ Fallback Analysis Complete.")
    except Exception as fallback_err:
        print(f"   ❌ Fallback failed: {fallback_err}")
        print("   Trying Document Intelligence Client as last resort...")
        if hasattr(doc_client, "begin_analyze_document"):
             poller = doc_client.begin_analyze_document("prebuilt-layout", file_bytes)
             result = poller.result()
             print("   ✅ DocIntel Fallback Complete.")

print("\n✅ Analysis Complete. Extracting Information...")

# --- Extract and Display Markdown ---
contents = []
markdown_text = ""

if result:
    # Helper to normalize result
    if hasattr(result, "as_dict"):
        res_data = result.as_dict()
        # Try to get contents from multiple possible locations
        contents_list = res_data.get("contents") or res_data.get("result", {}).get("contents", [])
        if contents_list:
            class PseudoContent:
                 def __init__(self, c_dict): 
                     self.markdown = c_dict.get("markdown", "")
                     self.kind = c_dict.get("kind", "")
            contents = [PseudoContent(c) for c in contents_list]
        elif "pages" in res_data:
            print("   (Processing Layout Model Output)")
            full_text = res_data.get("content", "")
            class PseudoContent:
                 def __init__(self, txt): self.markdown = txt
            contents = [PseudoContent(full_text)]
    

    if not contents and hasattr(result, "content"):
        class PseudoContent:
             def __init__(self, txt): self.markdown = txt
        contents = [PseudoContent(result.content)]

if contents:
    content = contents[0]
    markdown_text = content.markdown
    
    if markdown_text:
        print("=" * 50)
        print(markdown_text[:1000] + "...\n(truncated)") 
        print("=" * 50)
        
        if "Figure" in markdown_text or "image" in markdown_text.lower():
            print("\n🔎 FOUND FIGURES in Markdown!")
    else:
        print("⚠️ No direct 'markdown' field found.")
else:
    print("❌ No contents found even after fallback.")

# --- Save Results ---
if result:
    output_file = "content_understanding_result.json"

    def result_to_dict(obj):
        if hasattr(obj, "as_dict"): return obj.as_dict()
        if hasattr(obj, "__dict__"): return obj.__dict__
        if isinstance(obj, list): return [result_to_dict(i) for i in obj]
        return obj

    with open(output_file, "w") as f:
        json.dump(result_to_dict(result), f, indent=2, default=str)
    print(f"\n📋 Full result saved to: {output_file}")


### 💡 Why JSON instead of Markdown?

You might wonder: *"Why deal with this complex JSON object? Why not just use the Markdown text?"*

**1. Precise "Pointing" (Grounding)**
*   **Markdown**: It's just text. `![Figure](...)`
*   **JSON**: Contains the **Bounding Box** `[x, y, w, h]` and **Page Number**.
*   **Significance**: When the RAG bot answers a user question about the "Circuit Diagram," it can highlight the **exact region on the PDF** in the UI. You can't do that with plain Markdown.

**2. Structure Awareness**
*   **Markdown**: Headers are just `#`. It's hard to know if `# Intro` is a child of `# Chapter 1` without parsing the whole string.
*   **JSON**: The `paragraphs` list often contains parent/child relationships or explicit roles (`sectionHeading`), making **Semantic Chunking** much more robust.

**3. Multi-Element Data**
*   **Markdown**: Tables are converted to text pipes `| col | col |`. Complex merged cells break easily.
*   **JSON**: Tables are objects with rows, columns, spans, and logic. We can reconstruct them perfectly or ask an LLM to "read row 3".


In [ ]:
# --- 🔍 INSPECTION: PROVING THE "INVISIBLE" IS NOW VISIBLE ---
from IPython.display import Image, display
import os
import re

print(f"🔎 Inspecting analysis results for Figures (comparing to 'Invisible' Module 1)...\n")

# 1. VISUAL TARGET (What we missed in Module 1)
ref_img = "../module-1-naive-rag/page12.png"
if os.path.exists(ref_img):
    print("🎯 THE GOAL: Detect and Describe this Diagram (Page 12):")
    display(Image(filename=ref_img, width=400))
else:
    print(f"(Reference image {ref_img} not found locally)")

target_page = 12

# --- 2. CHECK MARKDOWN DESCRIPTION (Semantic Proof) ---
print("\n--- 📝 CHECKING NATIVE AI DESCRIPTION (from 'prebuilt-documentSearch') ---")

# The prebuilt-documentSearch returns markdown in this format:
# ![OCR_FALLBACK_TEXT](figures/12.1 "SEMANTIC_DESCRIPTION_HERE")
#
# The SEMANTIC description is in the TITLE attribute (after the URL, in quotes)
# The OCR text is just a fallback label for the alt-text

if "Example 2" in markdown_text:
    try:
        start_idx = markdown_text.find("Example 2")
        snippet = markdown_text[start_idx:start_idx+2000]  # Larger window
        
        # Look for the figure with BOTH alt-text AND title (semantic description)
        # Pattern: ![alt_text](url "title_description")
        match = re.search(r'!\[(.*?)\]\((.*?)\s+"(.*?)"\)', snippet, re.DOTALL)
        
        if match:
            alt_text = match.group(1)
            figure_url = match.group(2)
            semantic_description = match.group(3)
            
            print(f"✅ FOUND FIGURE WITH SEMANTIC DESCRIPTION:")
            print(f"   Figure URL: {figure_url}")
            print(f"   Alt-Text (OCR Fallback): \"{alt_text[:50]}...\"")
            print(f"\n   🎉 SEMANTIC DESCRIPTION (AI-Generated):")
            print(f"   \"{semantic_description[:500]}...\"")
            
            # Check if it's truly semantic (more than just OCR noise)
            if len(semantic_description) > 100 and ("axis" in semantic_description.lower() or "diagram" in semantic_description.lower() or "circuit" in semantic_description.lower()):
                print(f"\n   ✨ SUCCESS! prebuilt-documentSearch generated a rich semantic description!")
                print(f"      No client-side GPT-4o call was needed.")
                print(f"      The service did all the multimodal analysis internally.")
            else:
                print(f"\n   ⚠️ Description seems short or generic. May need review.")
        else:
            # Fallback: try the simpler pattern (no title)
            simple_match = re.search(r'!\[(.*?)\]\((.*?)\)', snippet)
            if simple_match:
                alt_text = simple_match.group(1)
                print(f"⚠️ FOUND IMAGE TAG (without semantic title):")
                print(f"   Alt-Text: \"{alt_text}\"")
                print(f"\n   This looks like Layout-mode output (OCR only).")
            else:
                print("   (Found 'Example 2' text, but no image tag nearby)")

    except Exception as e:
        print(f"   Error extracting markdown: {e}")
else:
    print("   'Example 2' text not found in markdown output.")

# --- 3. BONUS: Show Chart.js code if present ---
print("\n--- 📊 CHECKING FOR INTERACTIVE CHART CODE ---")
if "```chart" in markdown_text:
    chart_start = markdown_text.find("```chart")
    chart_end = markdown_text.find("```", chart_start + 8)
    if chart_end > chart_start:
        chart_code = markdown_text[chart_start:chart_end+3]
        print("✅ FOUND Chart.js code for interactive rendering!")
        print(f"   (First 200 chars): {chart_code[:200]}...")
        print("\n   💡 This means the service converted the V-I graph into Chart.js syntax!")
        print("      You could render this interactively in a web UI.")
else:
    print("   No Chart.js code found in output.")

## Lab 3.2: Semantic Chunking

Now let's look at text. In Module 1, we chopped text every 500 characters.
Here, we will use the **Document Map** from Document Intelligence to chunk by **Section Headers**.

This ensures that a whole topic stays together.

In [ ]:
print(f"Analyzing {PDF_PATH} (Layout Model)...")
# Re-run analysis for this module to be self-contained
with open(PDF_PATH, "rb") as f:
    poller = doc_client.begin_analyze_document("prebuilt-layout", f)
    result = poller.result()

print("Analysis complete.")

In [ ]:
# --- SEMANTIC CHUNKING LOGIC ---
# Strategy: 
# 1. Iterate through paragraphs.
# 2. If we hit a 'sectionHeading', start a new chunk.
# 3. Accumulate content until the next heading.

semantic_chunks = []
current_chunk = {"title": "Intro", "content": ""}

for p in result.paragraphs:
    role = p.role if p.role else "body"
    content = p.content
    
    if role == "sectionHeading":
        # Save previous chunk if it has content
        if current_chunk["content"].strip():
            semantic_chunks.append(current_chunk)
        
        # Start new chunk
        current_chunk = {"title": content, "content": ""}
    else:
        # Append to current chunk
        # Filter out page headers/footers here if desired (as learned in Mod 2)
        if role not in ["pageHeader", "pageFooter"]:
            current_chunk["content"] += content + "\n"

# Add last chunk
if current_chunk["content"].strip():
    semantic_chunks.append(current_chunk)

print(f"Created {len(semantic_chunks)} Semantic Chunks.")

# Inspect a few
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"--- Chunk {i+1}: {chunk['title']} ---")
    print(chunk['content'][:200].replace("\n", " ") + "...")
    print("-----------------------------------\n")

## Conclusion

We have upgraded our pipeline:
1. **Figures**: Are no longer invisible holes; they are text descriptions generated by **Content Understanding** (using GPT-4.1-mini internally).
2. **Text**: Is no longer arbitrary fragments; it is organized by **Topic** (Headings).

In **Module 6 (Retrieval)**, we will see how these improvements lead to vastly better search results.

In [ ]:
# --- 3. COMPARISON: FIXED VS SEMANTIC ---
print("\n--- 🔍 COMPARISON: Module 1 (Fixed) vs Module 3 (Semantic) ---")

# Let's take the "Objectives" section as an example
# In Semantic chunking, we captured it cleanly as "Chunk 2" above.
semantic_example = next((c for c in semantic_chunks if "Objective" in c['title']), None)

print(">>> [SEMANTIC CHUNK]:")
if semantic_example:
    print(f"Title: {semantic_example['title']}")
    print(f"Content Length: {len(semantic_example['content'])} chars")
    print(f"Start: '{semantic_example['content'][:50]}...'")
    print(f"End:   '...{semantic_example['content'][-50:]}'")
    print("\n✅ RESULT: The entire 'Objectives' list (1, 2, 3...) is kept together.")
else:
    print("Objectives chunk not found.")

print("\n>>> [NAIVE CHUNK SIMULATION (500 chars)]:")
# Let's simulate what happens if we cut that same text at 500 chars randomly
if semantic_example:
    text = semantic_example['content']
    # Imagine the naive chunk started 50 chars before this section and cut off at 500
    simulated_naive_chunk = text[100:600] 
    print(f"Start: '...{simulated_naive_chunk[:50]}...'")
    print(f"End:   '...{simulated_naive_chunk[-50:]}...'")
    print("\n❌ RESULT: It likely starts in the middle of a sentence and cuts off point #3.")
    print("   If you search for 'What is objective #4?', the Naive chunk might miss it completely.")